gRPC è un framework open-source e high-performance per la comunicazione RPC, che supporta diversi linguaggi di programmazione e può runnare su diverse piattaforme. Usato da Netflix, Square, Cisco etc..

Abbiamo visto con Go RPC una calcolatrice dove il client, con `Arithmetic.Multiply`, mandava due numeri al server e riceveva il prodotto.  
Con **gRPC** vogliamo fare la stessa cosa, ma **immaginando che il client sia scritto in due linguaggi diversi** (es. client in Java, server in Go).

Per fare qualcosa del genere l'idea è dare a entrambi lo stesso "contratto", un file `.proto`, dove si dichiarano il metodo, la forma della richiesta e la forma della risposta affinché si possa generare il codice che permetta al client Java di inviare la richiesta e al server Go di capirla.  

Questa possibilità è molto importante nelle architetture a microservizi, dove servizi diversi possono essere sviluppati da team diversi e scritti in linguaggi differenti a seconda dell'esigenza: si parla di applicazioni **poliglotte**.

### HTTP/2
Il protocollo gRPC invia le richieste e le risposte tramite protocollo applicativo **HTTP/2**. Ripassiamolo.

Immaginiamo che un client mandi due richieste, $A$ e $B$, sulla stessa connessione: la prima che avvia un lavoro che richiede 2 sec e la seconda che chiede un dato già pronto nel server.  
Con il *pipelining* di HTTP/1.1 il client può spedire entrambe senza aspettare la prima risposta, ma il server deve rispondere nell'ordine in cui ha ricevuto le richieste --> **head-of-line blocking**.  

Con HTTP/2 invece possiamo assegnare $A$ a uno stream e $B$ a un altro (dove lo stream è uno scambio logico dentro la connessione TCP: i dati sono suddivisi in frame e ogni frame ha un identificativo di stream) --> il server può inviare i frame della risposta $B$ non appena sono pronti, senza aspettare la risposta $A$. Può anche alternare i frame di $A$ e $B$.

<img src="imgs/gRPC1.png" width="400">

Nota come tutti gli stream condividono la stessa connessione TCP --> **multiplexing**; si possono avere più scambi in corso sulla stessa connessione.  
L'head of line blocking è risolto da HTTP/2 **solo a livello applicativo**: se un segmento TCP si perde, dato che gli stream condividono la stessa connessione TCP --> possono risentire tutti di quella perdita. Il blocco a livello TCP quindi rimane con HTTP/2 e viene risolto solo con HTTP/3 e QUIC.

Quindi tra i principali vantaggi che gRPC trae nell'uso di HTTP/2:
- **multiplexing**: più richieste e risposte possono essere inviate contemporaneamente sulla stessa connessione TCP
- **header compression**: invece di inviare ripetutamente uno stesso header, HTTP/2 invia solo il delta riducendo così l'overhead
- **controllo del flusso**: si limita la quantità di dati che possono essere inviati dal mittente sia sul singolo stream che sulla connessione TCP complessiva
- **full duplex**: client e server possono inviare dati contemporaneamente, senza dover aspettare che l'altro finisca di inviare

### Protocol Buffer e file `.proto`
Ora che abbiamo visto come viaggiano i dati, vediamo di capire come fa il server a sapere che cosa significano i byte che riceve.  
In gRPC si usa un linguaggio di definizione dei dati chiamato **Protocol Buffer** (o **ProtoBuf**). Questo svolge due ruoli principali:
1. come **IDL** (Interface Definition Language) per definire i messaggi e i servizi che il client e il server si scambiano (i.e. definire il contratto) tramite il file `.proto`
2. come **formato di serializzazione**, stabilendo cioè come i valori di quei messaggi vengono codificati in byte quando parte una chiamata. In particolare la serializzazione è **binaria** e quindi molto compatta e veloce.

In questo senso gRPC segue un **proxy pattern**: il client e il server non si scambiano direttamente i messaggi, ma lo fanno tramite un **proxy** (client stub e skeleton nel server) che si occupa di serializzare e deserializzare i messaggi (esattamente come visto in Sun RPC).

Per capire come funziona, consideriamo un messaggio `Person`:
```go
message Person {
    string name = 1;
    int32 id = 2;
    string email = 3;
}
```
`string` è il tipo, `name` è il nome del campo e `1` **è il numero con cui quel campo viene identificato nel messaggio binario**.  
Ad es. immaginiamo di mandare un messaggio `Person` con `name = "Anna"`, `id = 123` e `email = "anna@example.com"` --> quando il destinatario legge il messaggio binario e vede il numero `1` sa che dovrà interpretare i byte successivi come una stringa e assegnarla al campo `name` e lo stesso vale per gli altri campi.

Quindi un numero identifica il campo all'interno di un singolo messaggio --> in messaggi diversi lo stesso numero può essere usato per campi diversi.

Qui quindi differenza sostanziale rispetto a `gob` in Go RPC, dove le descrizioni dei tipi erano trasportate direttamente nel flusso di messaggi, si parlava di serializzazione **self-describing** che invece in gRPC si evita grazie al ProtoBuf.

Vediamo un esempio completo di interfaccia di servizio `ProductInfo`; anzitutto si definiscono le interfacce dei metodi, poi i campi dei messaggi considerati:
```go
service ProductInfo {
    rpc AddProduct(Product) returns (ProductID);
    rpc GetProduct(ProductID) returns (Product);
}

message Product {
    string id = 1;
    string name = 2;
    float price = 3;
}

message ProductID {
    string id = 1;
}

```

I passaggi in pratica sono i seguenti:
1. **si scrive il contratto nel file `.proto`**
2. **si genera il codice di comunicazione** (client stub e server skeleton) a partire dal file `.proto` **tramite il compilatore** `protoc` (è qui la somiglianza con Sun RPC).  
E per linguaggi diversi? Si usano plugin differenti: ad es. per Go `run protoc` per generare manualmente gli stub, per Java si usano strumenti come Maven o Gradle etc...
1. **si scrive il comportamento dell'applicazione** tramite API gRPC per il linguaggio specifico. Per gli esempi, considereremo gRPC-Go.

<img src="imgs/gRPC2.png" width="400">

<img src="imgs/gRPC3.png" width="400">

#### helloworld example in gRPC-Go
Vediamo un esempio dove il client manda un nome e il server deve rispondere con "Hello (persona)", con sia client che server scritti in Go.

1- **Definizione del contratto**: 

Nel `.proto` dichiariamo il servizio e i messaggi coinvolti:
```go
service Greeter {
    rpc SayHello(HelloRequest) returns (HelloReply);
}

message HelloRequest {
    string name = 1;
}

message HelloReply {
    string message = 1;
}
```

2- **Generazione del codice**:

Quando eseguiamo `protoc` per Go otteniamo due file:
- `helloworld.pb.go`: contiene struct Go per i tipi dei messaggi `HelloRequest` e `HelloReply`, e i metodi per serializzare e deserializzare i messaggi
- `helloworld_grpc.pb.go`: permette di fare e ricevere la chiamata; da qui arrivano `pb.NewGreeterClient` e `pb.RegisterGreeterServer`.

dove `pb` è l'alias del package generato dal compilatore protoc.

3_1- **Implementazione del server**:

Chi scrive il server anzitutto deve scrivere cosa deve fare SayHello:
```go
func (s *server) SayHello(
    ctx context.Context, 
    in *pb.HelloRequest,
    ) (*pb.HelloReply, error) {
        //implement service logic here
    }
```
per capirlo immaginiamo che sia arrivata la richiesta con nome "Anna" --> `in` contiene il messaggio `HelloRequest` con il campo `name` valorizzato a "Anna" --> con `in.GetName()` otteniamo "Anna" e possiamo costruire il messaggio di risposta `HelloReply` scrivendo tipo:
```go
return &pb.HelloReply{Message: "Hello " + in.GetName()}, nil
```
Qui si vede chiaramente la differenza rispetto a Go RPC: `Multiply` riceveva un puntatore alla risposta da riempire e restituiva solo l'errore. Qui invece `SayHello` restituisce la risposta insieme all'errore.

`ctx` è il contesto della chiamata, che permette di passare informazioni tra client e server (es. timeout, metadati, ecc..). Es. il client può concedere alla RPC 100 ms, questa scadenza viene quindi comunicata anche al server che può tenerne conto mentre lavora. Se la chiamata viene cancellata, il framework ne segnala la cancellazione proprio grazie al contesto e il server può interrompere il lavoro in corso.

A questo punto bisogna **mettere il servizio in ascolto**:
```go
lis, err := net.Listen("tcp", port)
if err != nil {
    log.Fatalf("failed to listen: %v", err)
}
s := grpc.NewServer()
pb.RegisterGreeterServer(s, &server{})
s.Serve(lis)
```
Si apre la porta con `net.Listen`, si istanzia un server gRPC con `grpc.NewServer()` e si registra il servizio con `pb.RegisterGreeterServer`. Infine si mette in ascolto con `s.Serve(lis)`.

3_2- **Implementazione del client**:

Il client prepara il canale gRPC verso l'indirizzo del server, e poi usa quel canale per creare lo stub del servizio `Greeter`. Supponendo che `conn` sia il canale e `ctx` il contesto della chiamata:
```go
c := pb.NewGreeterClient(conn)
r, err := c.SayHello(ctx, &pb.HelloRequest{Name: "Anna"})
```
La prima riga prepara `c`, lo stub locale. Nella seconda accade lo scambio effettivo, i.e.:
1. il client passa allo stub una richiesta contenente "Anna"
2. lo stub la codifica con **Protobuf** e la invia al server tramite **HTTP/2**
3. Il server riceve la richiesta ed esegue il `SayHello` che abbiamo scritto
4. La risposta torna al client, lo stub la decodifica, e se `err` è `nil` allora `r` contiene il messaggio di risposta.

##### Aggiungere un metodo al servizio
Supponiamo che `Greeter` offra già `SayHello` e che vogliamo aggiungere il metodo `SayHelloAgain`. Per prima cosa dobbiamo aggiornare il file `.proto` aggiungendo la nuova RPC:
```go
rpc SayHelloAgain(HelloRequest) returns (HelloReply);
```
dal momento che abbiamo usato gli stessi messaggi `HelloRequest` e `HelloReply`, non dobbiamo aggiungerne di nuovi. 

Quindi a questo punto si rigenera il codice con `protoc`, si implementa il nuovo metodo nel server e si aggiorna il client per chiamare il nuovo metodo.

----

In generale, esistono quattro modi per scambiare messaggi in una RPC.  
Del resto, con HTTP/2, abbiamo visto come lo stream possa trasportare dati in entrambe le direzioni.

Immaginiamo un servizio di mappe, dove a volte ho bisogno di chiedere info su *un punto* e altre *di inviare tutti i punti di un percorso man mano che li percorro*, oppure di ricevere *molti punti in una sola ricerca*, è qui che l'uso degli stream diventa utile:
1. **Simple (unary) RPC**: il client invia una singola richiesta e aspetta una singola risposta, sarebbe il corrispettivo di una singola chiamata di funzione.  
`rpc GetFeature(Point) returns (Feature) {}`
2. **Server streaming RPC**: il client invia una singola richiesta e riceve un flusso di risposte, quindi il client legge fino alla fine dello stream e l'ordine è garantito. Nell'esempio di mappe, il client invia un rettangolo e riceve tutti i punti di interesse che si trovano all'interno del rettangolo.  
`rpc ListFeatures(Rectangle) returns (stream Feature) {}`
3. **Client streaming RPC**: il client invia un flusso di richieste e riceve una singola risposta. Anche qui gRPC garantisce l'ordine dei messaggi dentro ogni chiamata RPC. Un esempio è se il client invia un flusso di punti e il server risponde con un sommario del percorso.  
`rpc RecordRoute(stream Point) returns (RouteSummary) {}`
4. **Bidirectional streaming RPC**: sia il client che il server inviano flussi di messaggi **contemporaneamente** (full duplex!). Il server può aspettare di ricevere tutti i messaggi del client, oppure possono fare "ping-pong" (i.e. il server può rispondere a un messaggio prima che il client abbia finito di inviare tutti i messaggi). Ancora ordine preservato nello stream di ogni chiamata RPC. Un esempio è se il client invia un flusso di punti e il server risponde con un flusso di note sul percorso durante la stessa chiamata  
`rpc RouteChat(stream RouteNote) returns (stream RouteNote) {}`

Prendiamo `ListFeatures` e ipotizziamo che il server trovi nell'area una scuola, una stazione e un parco --> può inviare tre msg distinti nella stessa RPC. Qui si vede la differenza tra streaming e multiplexing: lo streaming riguarda più messaggi in una singola RPC, mentre il multiplexing riguarda più RPC sulla stessa connessione TCP.

Trasparenze di gRPC:
- **Accesso**: quando si scrive un metodo (es. `client.GetFeature(ctx, point)`) si sta chiamando un metodo sullo **stub locale**, che fa da proxy e si occupa di serializzare, inviare, ricevere e deserializzare i messaggi. Qui trasparenza ancora più forte perché funziona anche se l'altro servizio è scritto in un linguaggio diverso. Comunque quindi trasparenza all'accesso parziale, dato che il client può osservare errori remoti/scadenze.
- **Locazione**: negli esempi visti il client deve sempre sapere **dove collegarsi** (es. `localhost:50051`) --> lavora tramite risoluzione DNS, non c'è direttamente un meccanismo di **service discovery** che tenga traccia di quali istanze offrono un servizio.  
Esiste comunque la possibilità di integrare resolver personalizzati con sistemi come Consul o Eureka e una service mesh come Istio...
- **Concorrenza**: il framework permette a più chiamate di essere in corso contemporaneamente. Come al solito però l'accesso concorrente a risorse condivise va gestito dal programmatore --> parzialmente trasparente
- **Replicazione**: si integra tramite **load balancers** che siano client-side o esterni (es. di lb esterno: Envoy).  
Ma avendo istanze diverse dello stesso servizio che magari conservano dati, come si gestisce la consistenza? Due possibilità:
   1. **Replicare lo stato**: più istanze conservano lo stesso stato e quindi rispondono allo stesso modo alle stesse richieste --> si può raggiungere qualunque replica a condizione che il sistema propaghi e coordini gli aggiornamenti di consistenza
   2. **Partizionare lo stato**: ogni istanza conserva solo una parte dello stato e quindi risponde solo a una parte delle richieste
- **Fallimento**: gRPC di default ha semantica **maybe** ma con retry trasparenti, nel senso che la libreria può ritentare automaticamente nei casi in cui sa che il codice applicativo del server non ha ancora gestito la chiamata (es. se la richiesta non è uscita dal client o è stata rifiutata prima di arrivare al metodo).  
Lo sviluppatore può però configurare come si vede tra poco una strategia di ritentativi tramite ad esempio *backoff exp.*, che porta con sé una semantica **at-least-once**.  
Sta poi a carico dello sviluppatore aggiungere la parte di deduplicazione lato server per ottenere anche la **at-most-once**.  
Si ricorda il limite sull'**exactly-once**: difficile da implementare, e se non lo si ha c'è il rischio che un metodo si interrompa a metà lasciando alcune modifiche già applicate...

Per quanto riguarda la trasparenza di fallimento in particolare due strategie che si possono implementare lato client (non sono automatiche, sono un'opzione di gRPC da configurare, del resto metterle porta a un possibile overhead):
- **Retry con backoff exp**: un tentativo fallisce --> il client riprova dopo un'attesa che cresce in modo exp
- **Request hedging**: se un primo tentativo è ancora in corso ma è abbastanza lento da giustificare un secondo tentativo, il client invia una seconda richiesta magari verso un'altra replica. Quando arriva la risposta, il client può cancellare i tentativi ancora in corso.  

<img src="imgs/gRPC4.png" width="400">

Altra trasparenza presente in gRPC è quella legata alla **sicurezza**: gRPC supporta **SSL/TLS** per cifrare le comunicazioni e autenticare i peer.

Con TLS il server presenta un certificato e il client lo verifica tramite una fonte di fiducia configurata, e la comunicazione è cifrata. Si può usare in gRPC anche mTLS (mutual TLS) dove anche il client presenta un certificato e il server lo verifica.  
Inoltre si possono usare **token di comunicazione** che il client può inviare nei metadati di una chiamata per dimostrare la propria identità e autorizzazioni a lvl applicativo.

Con TLS quindi si garantisce **confidenzialità**, **integrità** e **autenticazione del server**.

Per misurare le prestazioni di gRPC, Google ha sviluppato un benchmark chiamato **ghz** che permette di simulare carichi di lavoro e misurare latenza e throughput --> ha senso usarlo magari per vedere come reagisce l'autoscaling quando aumenta il carico.

Limiti pratici:
- dal browser si usa normalmente gRPC-Web, che però ha un supporto più limitato rispetto a un client gRPC nativo
- il formato binario di Protobuf è si compatto e veloce, ma non human-readable come JSON o XML, serve decodificarlo
- utilizzato soprattutto in sistemi grandi e complessi, dove si deve organizzare discovery, balancing e gestione del traffico tra servizi.

Dalla prossima lezione vediamo code di msg e pub/syb, che permettono di scambiare msg passando attraverso un intermediario (mentre RPC collega direttamente chi chiama un servizio)